In [1]:
import pandas as pd
import numpy as np

# read in the metadata file 
metadata = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/zenodo/cell_metadata_cols.tsv", sep="\t")

# read in file containing bam file IDs to match with metadata
ftp_data = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/RAW/filereport_read_run_PRJEB14362_tsv.txt", sep="\t")
# Extract the cell name from the `submitted_ftp` column
ftp_data["cell_name"] = ftp_data["submitted_ftp"].str.split(";").str[0].str.split("/").str[-1].str.split(".").str[0]

In [2]:
import os
# Paths
juncs_path = "/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/LEAFLET"
output_path = "/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/LeafletFA/ATSEs"
gtf_file = None
# gtf_file = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/human-reference/gencode/gencode.v45.primary_assembly.annotation.gtf"
using_annotations = "NO"

# Get all files in juncs_path that end in *junctions_with_barcodes.bed
junc_files = [os.path.join(juncs_path, x) for x in os.listdir(juncs_path) if x.endswith("junctions_with_barcodes.bed")]
junc_files[1]

'/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/LEAFLET/ERR2946920_junctions_with_barcodes.bed'

In [37]:
for col in metadata.iloc[1]:
    print(col)

1
aux_info
True
21843_1#100
0.99945606112355
day1
fafq
IU
expt_09
1001
True
False
False
False
IU
IU
5.34744674008841
5.28937682161842
3.09271056492959
4.4449177529445
4.42519115768671
3.96553094362286
3.96411814315148
1.25527250510331
1.49136169383427
1.14612803567824
mapping
data_raw/scrnaseq/run_21843/fastq/21843_1#100_1_val_1.fq.gz
data_raw/scrnaseq/run_21843/fastq/21843_1#100_2_val_2.fq.gz
2515
9200
237159.0
4096
0
237030
881010
46858
1
237159.0
338590.0
180345
2315
87.4842357749707
0.555795239969125
12.5157642250293
11.9599689850602
39.2475939746694
33.2810820797494
100.0
50.2818599092326
44.6640784206686
28.4919048281731
22.4450484561957
100.0
100.0
64.2320077648003
59.8233785358838
70.0431199976373
126
0126_E04
1.0
SCGC--0126_E04
( data_raw/scrnaseq/run_21843/fastq/21843_1#100_1_val_1.fq.gz, data_raw/scrnaseq/run_21843/fastq/21843_1#100_2_val_2.fq.gz )
0.8.2
none
21843_1#100
True
0.50733333372292
Sat Nov 18 06:41:59 2017
0.493105640117592
222558.809076966
194703.87327086
1236.97

In [40]:
ftp_data.iloc[1]

run_accession                                                          ERR1588105
sample_accession                                                     SAMEA4082672
experiment_accession                                                   ERX1658751
study_accession                                                        PRJEB14362
tax_id                                                                       9606
scientific_name                                                      Homo sapiens
instrument_platform                                                      ILLUMINA
instrument_model                                              Illumina HiSeq 2000
library_name                                                             17120970
library_strategy                                                          RNA-Seq
library_selection                                                            cDNA
last_updated                                                           2018-11-16
experiment_title

In [6]:
# add a column to ftp_data whihc checks whether cell_name is in metadata["cell_name"]
ftp_data["cell_in_metadata"] = ftp_data["cell_name"].isin(metadata["cell_name"])
# do the same for metadata
metadata["cell_in_ftp"] = metadata["cell_name"].isin(ftp_data["cell_name"])

In [29]:
ftp_data["cell_in_metadata_public_name"] =  ftp_data["sample_title"].isin(metadata["public_name"])
# filter ftp_data to only include cell_in_metadata == True or cell_in_metadata_public_name == True
ftp_data[(ftp_data["cell_in_metadata"] == True) | (ftp_data["cell_in_metadata_public_name"] == True)].shape, ftp_data[(ftp_data["cell_in_metadata"] == True)].shape

((25334, 21), (23793, 21))

In [33]:
# check which metadata["public_name"] values match with ftp_data["sample_title"] for the ones where metadata["cell_in_ftp"] is False
metadata["cell_in_ftp_sample_title"] = metadata["public_name"].isin(ftp_data["sample_title"])
# check where cell_in_ftp is False but cell_in_ftp_sample_title is True
metadata[(metadata["cell_in_ftp"] == True) | (metadata["cell_in_ftp_sample_title"] == True)].shape, metadata[(metadata["cell_in_ftp"] == True)].shape, metadata.shape

((24582, 95), (23793, 95), (36044, 95))

In [ ]:
# ftp_data = ftp_data[["run_accession", "cell_name", "library_name", "sample_title"]].drop_duplicates()

In [ ]:
metadata = metadata.merge(ftp_data, on = "cell_name")
# make new column in metadata called cell_id using metadata["run_accession"] 
metadata["cell_id"] = metadata["run_accession"]

In [ ]:
metadata = metadata[["cell_id", "cell_name", "library_name", "donor", "day", "experiment", "plate_id", "plate_well_id", "donor_short_id", "donor_long_id"]].drop_duplicates()
metadata.head()

In [ ]:
WD = "/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/LeafletFA/ATSEs"

# save the metadata file to the working directory and add today's date 
import datetime
today = datetime.datetime.today().strftime('%Y-%m-%d')
metadata.to_csv(f"{WD}/metadata_{today}.tsv", sep="\t", index=False)